### **Severity-Weighted Accident Proportion per Municipality**

1. Add +1 to the `severity` column
2. Compute, for each municipality, the proportion of accidents for every year (weigthed by severity)?
3. Try Linear Extrapolation for 2025-2026, understand how it works

In [64]:
import pandas as pd

In [65]:
df = pd.read_csv("C:/Users/Horia/Desktop/KUL/Modern Data Analytics/cleaned_bike_accidents.csv")

**Step 1:** Adding +1 to the `severity` column to avoid zeros

In [66]:
df['severity'] = df['severity'] + 1

**Step 2: Accident Rate per Municipality, multiplied by `severity`**

In [67]:
flemish_df = df[(df['CD_NIS'] < 50000) & ~df['CD_NIS'].between(21000, 21999)]

accidents = (
    flemish_df.groupby(['DT_YEAR_COLLISION', 'TX_MUNTY_COLLISION_NL'])
    .agg(
        accident_count=('severity', 'size'),
        avg_severity=('severity', 'mean')
    )
    .reset_index()
    .rename(columns={
        'DT_YEAR_COLLISION': 'year',
        'TX_MUNTY_COLLISION_NL': 'municipality'
    })
)

accidents['accident_score'] = accidents['accident_count'] * accidents['avg_severity']


In [68]:
accidents

,year,municipality,accident_count,avg_severity,accident_score
0,2017,Aalst,96,1.135417,109.0
1,2017,Aalter,41,1.170732,48.0
2,2017,Aarschot,24,1.083333,26.0
3,2017,Aartselaar,30,1.033333,31.0
4,2017,Affligem,6,1.000000,6.0
...,...,...,...,...,...
2214,2024,Zuienkerke,3,1.000000,3.0
2215,2024,Zulte,7,1.000000,7.0
2216,2024,Zwalm,10,1.300000,13.0
2217,2024,Zwevegem,25,1.120000,28.0


**Step 3:** Linear Extrapolation to obtain an `accident_score` per municipality for **2025** and **2026**

Linear extrapolation fits a straight line (Ordinary Least-Squares) through the historical data points, for each municipality, it then simply extends that line into the future (2025, 2026)

**Key Assumption**: Whatever trend existed in 2017–2024 (rising, falling, flat) will continue at the same rate into 2025–2026. If there was a sudden change (e.g. a policy intervention, COVID), the extrapolation will not capture that.


In [69]:
from scipy.stats import linregress

def extrapolate(group, target_years=[2025, 2026]):
    slope, intercept, _, _, _ = linregress(group['year'], group['accident_score'])
    return pd.DataFrame({
        'year': target_years,
        'municipality': group['municipality'].iloc[0],
        'accident_score': [intercept + slope * y for y in target_years]
    })

extrapolated = (
    accidents.groupby('municipality')
    .apply(extrapolate)
    .reset_index(drop=True)
)

accidents_full = pd.concat([accidents, extrapolated], ignore_index=True).sort_values(['municipality', 'year'])

C:\Users\Horia\AppData\Local\Temp\ipykernel_18280\1193365315.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extrapolate)


In [73]:
accidents_full[accidents_full['municipality'] == "Leuven"]

,year,municipality,accident_count,avg_severity,accident_score
970,2020,Leuven,223.0,1.053812,235.000000
1246,2021,Leuven,292.0,1.082192,316.000000
1526,2022,Leuven,238.0,1.050420,250.000000
1805,2023,Leuven,211.0,1.090047,230.000000
2080,2024,Leuven,208.0,1.057692,220.000000
2505,2025,Leuven,NaN,NaN,260.178571
2506,2026,Leuven,NaN,NaN,265.273810


Keeping only the years after (& including) 2020 in order to match our bike-traffic dataset

In [71]:
accidents_full = accidents_full[accidents_full['year'] >= 2020]